In [1]:
from src.providers.graph6 import Graph6Provider
from src.converters.graph_to_smiles import GraphToCarbonSMILESConverter


provider = Graph6Provider("data/graph8c.g6")
converter = GraphToCarbonSMILESConverter()

In [2]:
import pandas as pd
from rdkit import RDLogger

# Disable all RDKit warnings and errors
RDLogger.DisableLog('rdApp.*')

smiles_records = []

for metadata, raw_graph in provider:
    graph_id = metadata["id"]
    
    smiles = converter.convert(raw_graph)
    smiles_records.append({"id": graph_id, "smiles": smiles})
    
    if len(smiles_records) % 1000 == 0:
        print(f"Processed {len(smiles_records)} graphs...")

# Convert to pandas DataFrame and save
# df = pd.DataFrame(smiles_records)
# df.to_parquet("data/graph8cSmiles.parquet", engine='pyarrow')

Processed 1000 graphs...
Processed 2000 graphs...
Processed 3000 graphs...
Processed 4000 graphs...
Processed 5000 graphs...
Processed 6000 graphs...
Processed 7000 graphs...
Processed 8000 graphs...
Processed 9000 graphs...
Processed 10000 graphs...
Processed 11000 graphs...


In [3]:
from src.models.huggingface import ChemBert

device = "mps"
embedder = ChemBert(pooling="mean")

Loading DeepChem/ChemBERTa-77M-MTR onto cpu...


Loading weights:   0%|          | 0/53 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: DeepChem/ChemBERTa-77M-MTR
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 
regression.out_proj.bias        | UNEXPECTED |  | 
norm_std                        | UNEXPECTED |  | 
regression.out_proj.weight      | UNEXPECTED |  | 
norm_mean                       | UNEXPECTED |  | 
regression.dense.bias           | UNEXPECTED |  | 
regression.dense.weight         | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
embeddings_list = []
batch_size = 64

smiles_batch = []

for i, m in enumerate(smiles_records):
    graph_id = m["id"]
    smiles = m["smiles"]
    smiles_batch.append(smiles)

    if len(smiles_batch) % batch_size == 0:
        embeddings = embedder.embed(smiles_batch)
        embeddings_list.append(embeddings)
        # smiles_batch = []
        break

In [7]:
embedding = embeddings[:, 0, :]